In [2]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

/home/gtamo/MS_ML


In [3]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm.auto import tqdm  # auto -> notebook/VSCode widget
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import pyarrow.dataset as pads
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from get_library import get_df   # CDD Vault collection export
# from tdc.multi_pred import DTI

/home/gtamo/miniconda3/envs/ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


> could not load openbabel


In [24]:
## params — single source of truth in config/config.yaml.
## Loaded as a `config` namespace AND injected as globals, so both
## `config.RAW_PROTEOMICS_PATH` and bare `RAW_PROTEOMICS_PATH` work.
import yaml
from types import SimpleNamespace
with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = SimpleNamespace(**_cfg)
globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')

# -------------------
# Dropbox access — stream files over the ssh tunnel when DROPBOX_SSH_HOST is set
# (see wiki "Dropbox access"; the reverse tunnel must be up), else read local paths.
# -------------------
os.environ.setdefault('DROPBOX_SSH_HOST', DROPBOX_SSH_HOST)
os.environ.setdefault('DROPBOX_SSH_PORT', str(DROPBOX_SSH_PORT))
_use_ssh  = bool(DROPBOX_SSH_HOST)
_dbx      = fn.open_dropbox if _use_ssh else (lambda p: p)   # wrap Dropbox paths; local data/ paths stay bare
_fbx_open = fn.open_dropbox if _use_ssh else None
_fbx_glob = fn.glob_dropbox if _use_ssh else None


> loaded 52 params from config/config.yaml


## 0. Imports

In [25]:
%%time
## latest library straight from CDD Vault (collections AJ/AK), compound name + smiles only
if CHEMLIB_OVERWRITE:
    serac_df = (get_df(vault=7108, collections=['AK', 'AJ'], columns=['name','Batch Mol-Batch ID', 'smiles','Px_repetition(yes/no)','Px_validated_WT(yes/no)','Px_Ligase_dependent(yes/no)',
                                                                      'Px_NameLigase_dependent','Px_Target_info','Px_Target_interest' ])
                .rename(columns={'name': 'compound'}))
    serac_df.to_csv(CHEMLIB_PATH,sep=',',index=False)
else:
    serac_df = pd.read_csv(CHEMLIB_PATH)
serac_df = serac_df.drop_duplicates()
serac_df['Px_validated_WT(yes/no)'] = serac_df['Px_validated_WT(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]
serac_df['Px_Ligase_dependent(yes/no)'] = serac_df['Px_Ligase_dependent(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]
serac_df['Px_repetition(yes/no)'] = serac_df['Px_repetition(yes/no)'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0})  # ['', 'yes', 'no'] -> [NaN, 1, 0]

## extract list of validated and devalidated targets
l1 = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==0) & 
                       (serac_df['Px_Target_interest'].notnull())]['Px_Target_interest']))
devalidated_targets = list(set([s.split(' ')[0].upper() for x in l1 for s in x.split(';')]))

l2 = list(set(serac_df[(serac_df['Px_Ligase_dependent(yes/no)']==1) & 
                       (serac_df['Px_Target_interest'].notnull())]['Px_Target_interest']))
validated_targets = list(set([s.split(' ')[0].upper() for x in l2 for s in x.split(';')]))

serac_df.shape

resolved_collections=[('AK', 931035), ('AJ', 931034)]
requested_columns=['name', 'Batch Mol-Batch ID', 'smiles', 'Px_repetition(yes/no)', 'Px_validated_WT(yes/no)', 'Px_Ligase_dependent(yes/no)', 'Px_NameLigase_dependent', 'Px_Target_info', 'Px_Target_interest']

--- collection name=AK id=931035 ---


  AK: 25row [01:23,  3.35s/row]


  collection_rows=25

--- collection name=AJ id=931034 ---


  AJ: 9474row [01:26, 109.18row/s]

  collection_rows=9474

total_rows=9499
multi_batch_molecules=119 (each emitted >1 row — one per batch)
CPU times: user 2.22 s, sys: 281 ms, total: 2.5 s
Wall time: 2min 51s


(9412, 9)

In [6]:
## local params — controls + contaminants come from config/config.yaml (CONTROLS / CONTAMINANTS)
control_compounds = list(CONTROLS)                                  # YAML list of control SRB ids
contaminants      = list(pd.read_csv(CONTAMINANTS)['Molecule Name'])  # contaminant compounds to remove
fbx_independent   = list(serac_df[ (serac_df['Px_Ligase_dependent(yes/no)']==0) | (serac_df['Px_validated_WT(yes/no)']==0) ]['compound'].unique()) 
cm2rm             = fbx_independent + control_compounds + contaminants

print(f'> {len(control_compounds)} control + {len(contaminants):,} contaminant compounds (from config) + {len(fbx_independent)} fbx independent = {len(cm2rm)}')

> 2 control + 81 contaminant compounds (from config) + 502 fbx independent = 585


In [14]:
if DFRAW_OVERWRITE:
# if 1<3:
    # -------------------
    # Old data
    # -------------------

    ## Px part 1
    df_raw_20260429, MS20260429 = fn.load_proteomics_data(
        RAW_PROTEOMICS_PATH,
        CLEAN_PROTEOMICS_PATH,
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
    )

    ## Px part 2:
    df_raw_20260520, MS20260520 = fn.load_proteomics_data(
        _dbx(PX_20260520_DB),            # raw per-gene table (Database export) — was wrongly CDDVault
        _dbx(PX_20260520_CDDVAULT),      # metadata table (Vault export: SMILES, Collections)
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
        mode='cddvault',           # Collections recipe (drops PROTACs), join on 'Batch Molecule-Batch ID'
        collections=['AJ', 'AK'],
    )

    ## Px part 3:
    df_raw_20260529, MS20260529 = fn.load_proteomics_data(
        _dbx(PX_20260529_DB),            # raw per-gene table (Database export) — was wrongly CDDVault
        _dbx(PX_20260529_CDDVAULT),      # metadata table (Vault export: SMILES, Collections)
        drop_plates=['Plate12', 'Plate15', 'Plate23'],
        mode='cddvault',           # Collections recipe (drops PROTACs), join on 'Batch Molecule-Batch ID'
        collections=['AJ', 'AK'],
    )
    # manual formatting for 20260529 metadata:
    MS20260529 = MS20260529.rename(columns={'Molecule-Batch ID':'Batch Molecule-Batch ID','Nr. Down':'MSData - Proteomics activities: Nr. Down',"Cmpd Activity" : "MSData - Proteomics activities: Cmpd Activity"})
    parts = MS20260529['Batch Molecule-Batch ID'].str.split('-', n=2, expand=True)
    MS20260529['Molecule Name'] = parts[0] + '-' + parts[1]   # 'SRB-0000385'
    MS20260529['batch']    = parts[2]                    # '001'

    ## Making final df_raw — tag each tranche with its date for the filter below
    df_raw = pd.concat([
        df_raw_20260429.assign(date=pd.to_datetime('20260429')),
        df_raw_20260520.assign(date=pd.to_datetime('20260520')),
        df_raw_20260529.assign(date=pd.to_datetime('20260529')),
    ]).reset_index(drop=True)
    # Per compound: keep the latest batch, then the latest date for repeats
    df_raw = fn.keep_latest_batch_per_compound(df_raw)
    df_raw[['genes']].drop_duplicates().to_csv('data/MS/Px_genes.csv')

    ## check whether all compounds are including in the df_raw:
    colskeep = ['Molecule Name','MSData - Proteomics activities: Nr. Down','origin',"MSData - Proteomics activities: Cmpd Activity"]
    MS = pd.concat([MS20260429.assign(origin='MS20260429'),
                    MS20260520.assign(origin='MS20260520'),
                    MS20260529.assign(origin='MS20260529')]).reset_index(drop=True)[colskeep].rename(columns={'Molecule Name':'compound',
                                                                                                            'MSData - Proteomics activities: Nr. Down':'ndown',
                                                                                                            "MSData - Proteomics activities: Cmpd Activity":'activity'})
    MS['date'] = pd.to_datetime(MS['origin'].str.replace('MS', ''))

    del df_raw_20260429,df_raw_20260520,df_raw_20260529

    # -------------------
    # New (FBX) data
    # -------------------

    # every tranche folder under FBX_DIR (config.yaml), named by export date
    TRANCHES = sorted(fn.glob_dropbox(FBX_DIR, '[0-9]*') if _use_ssh
                      else glob(os.path.join(FBX_DIR, '[0-9]*')))
    _fbx = {os.path.basename(t): fn.load_fbx_tranche(t, control_compounds=control_compounds,
                                                    contaminants=contaminants,
                                                    opener=_fbx_open, lister=_fbx_glob) for t in TRANCHES}

    # stack tranches; on a recurring experiment (uniquecontrast) / compound, keep the LATEST export
    df_raw_fbx = (pd.concat([d.assign(_tranche=k) for k, (d, _) in _fbx.items()], ignore_index=True)
                .sort_values('_tranche').drop_duplicates(['uniquecontrast', 'pg'], keep='last'))  # row key is (contrast, protein); deduping on contrast alone collapsed the proteome
    # stamp each FBX row with its tranche date (folder YYYYMMDD) — FBX_DFRAW_COLS carries no 'date',
    # so without this keep_latest_batch_per_compound sees NaT and drops every FBX row (NaT==NaT is False).
    df_raw_fbx['date'] = pd.to_datetime(df_raw_fbx['_tranche'].str[:8], format='%Y%m%d')
    df_raw_fbx = df_raw_fbx[fn.FBX_DFRAW_COLS + ['date']].reset_index(drop=True)
    MS_fbx = (pd.concat([m for _, m in _fbx.values()], ignore_index=True)
            .sort_values('date').drop_duplicates('compound', keep='last')
            [fn.FBX_MS_COLS].reset_index(drop=True))

    # align to the LIVE df_raw / MS schemas so a plain concat works regardless of any extra
    # columns the in-memory frames carry (e.g. smiles); columns FBX can't fill become NaN.
    _miss_dr = [c for c in df_raw.columns if c not in df_raw_fbx.columns]
    _miss_ms = [c for c in MS.columns     if c not in MS_fbx.columns]
    df_raw_fbx = df_raw_fbx.reindex(columns=df_raw.columns)
    MS_fbx     = MS_fbx.reindex(columns=MS.columns)
    print(f'\n> FBX formatted: df_raw_fbx {df_raw_fbx.shape} | MS_fbx {MS_fbx.shape}')
    print(f'  df_raw schema : {list(df_raw.columns)}')
    print(f'  MS schema     : {list(MS.columns)}')
    if _miss_dr: print(f'  [warn] df_raw cols NOT populated from FBX (set NaN): {_miss_dr}')
    if _miss_ms: print(f'  [warn] MS cols NOT populated from FBX (set NaN): {_miss_ms}')

    # -------------------
    # Merging
    # -------------------

    ## add df_raw:
    df_raw = pd.concat([df_raw, df_raw_fbx], ignore_index=True).drop_duplicates()
    df_raw = fn.keep_latest_batch_per_compound(df_raw)   # <-- add: collapse batches post-FBX too

    ## add the two MSs together
    MS = pd.concat([MS, MS_fbx], ignore_index=True)

    ## keep latest measurement with latest date
    MS = fn.collapse_ms_latest_measurement(MS)

    ## 2026-06-01 holds a single compound — fold it into the 2026-06-16 tranche
    MS.loc[MS['date'] == pd.Timestamp('2026-06-01'), 'date'] = pd.Timestamp('2026-06-16')

    ## remove contaminants and controls
    # MS = MS[~MS['compound'].isin(control_compounds + contaminants)]

    ## Ensure the compounds are FBX glues (no PROTACS) — same filter on df_raw for symmetry
    MS     = MS[MS['compound'].isin(serac_df['compound'])]
    df_raw = df_raw[df_raw['compound'].isin(serac_df['compound'])]
    MS.shape

    ## write to disk:
    # MS.to_parquet(MS_PATH, index=False)
    # df_raw.to_parquet(DFRAW_PATH, index=False)

else:
    MS     = pd.read_parquet(MS_PATH)
    df_raw = pd.read_parquet(DFRAW_PATH)


In [15]:
%%time
## Get smiles from library and compute MF features
print('> checking smiles')
# serac_df = serac_df.dropna().reset_index(drop=True)
failed_CMs = rdkit_tools.check_smiles_RDKiT(serac_df)
serac_df = serac_df[~serac_df['compound'].isin(serac_df)]
## then compute features:

print('> FEATURES_TYPE:',FEATURES_TYPE)

if FEATURES_TYPE == 'prevalence':
    MF_features_ = rdkit_tools.compute_H236_features(serac_df, v=True)
    _morgan = [c for c in MF_features_.columns if c.startswith('F') and c[1:].isdigit()]
    _drop   = [c for c in _morgan if MF_features_[c].mean() <= 0.02]
    MF_features = MF_features_.drop(columns=_drop).copy()

elif FEATURES_TYPE == 'autoresearch':
    with open('autoresearch/optimizeMS_genes_R2/logs/inputs_multifp.pkl', 'rb') as fh:
        MF_features = pickle.load(fh)['MF_features']

elif FEATURES_TYPE == 'H236':
    MF_features = rdkit_tools.compute_H236_features(serac_df, v=True)

else:
    properties  = rdkit_tools.compute_properties_from_smiles(serac_df)
    MF_features = pd.merge(MF,properties)

MF          = rdkit_tools.get_MF_bits_from_df(serac_df,nBits=2048)
MF_features = MF_features.drop_duplicates()
MF_features.head(1)

> checking smiles


  0%|          | 0/8936 [00:00<?, ?it/s]

100%|██████████| 8936/8936 [00:02<00:00, 3162.22it/s]


> FEATURES_TYPE: prevalence


H236 features: 100%|██████████| 8936/8936 [00:14<00:00, 624.00it/s]


CPU times: user 19.8 s, sys: 257 ms, total: 20 s
Wall time: 19.9 s


,compound,F0,F1,F5,F13,F29,F35,F36,F42,F43,...,AP_2038,AP_2039,AP_2040,AP_2041,AP_2042,AP_2043,AP_2044,AP_2045,AP_2046,AP_2047
0,SRB-0001195,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,1,1,1,0


## 1. Predictions

#### 1.1. Multiclass classification

In [42]:
test_0 = (pd.read_csv('tmp/2026-07-10-All AJ compounds with v2 as main vector.csv')
          .rename(columns={'Molecule Name':'compound','MSData - Proteomics activities: Cmpd Activity':'label'})
        [['compound','label']].dropna().drop_duplicates().reset_index(drop=True))

duplicated_compounds = list(set(test_0[test_0['compound'].duplicated(False)]['compound']))
print('> n duplicated cmps:',len(duplicated_compounds))

test_0 = test_0[~test_0['compound'].isin(duplicated_compounds)]

# combine single and low
_grp = {'Single (1)': 'Single/Low (1-10)', 'Low (2-10)': 'Single/Low (1-10)'}
test_0 = test_0.rename(columns={'activity':'label'}).assign(label=lambda d: d['label'].replace(_grp))

test_2 = pd.merge(serac_df,test_0)[['compound','label','smiles']].dropna().drop_duplicates()
test_2 = test_2[test_2['compound'].isin(MS['compound'])].reset_index(drop=True)
enum = test_2.drop('label',axis=1).drop_duplicates().reset_index(drop=True)
print(test_2.shape,enum.shape)
enum.head(1)

> n duplicated cmps: 73
(1322, 3) (1322, 2)


,compound,smiles
0,SRB-0000385,O=C(OCC1C2=CC=CC=C2C2=CC=CC=C12)N1[C@H]2CC3=CC...


In [43]:
## compute the FULL H236 feature universe.
MF_features_ = rdkit_tools.compute_H236_features(enum, v=True)
MF_features_ = enum[['compound', 'smiles']].merge(MF_features, on='compound').drop_duplicates('compound')

H236 features:   0%|          | 0/1322 [00:00<?, ?it/s]

H236 features: 100%|██████████| 1322/1322 [00:01<00:00, 665.18it/s]


In [40]:
## Predicting labels
## Score the enumeration with the deployable 4-class activity model (trained in MS_TargetML).
## MF_features above = the enum H236 feature universe (compound, smiles, + H236 cols).
model_path = 'output/ML/trained_models/20260709_Multi_Class/MultiClass_SSLMH_RF.joblib'
_b = joblib.load(model_path)
_rf, _fc, _cls = _b['model'], _b['feature_cols'], list(_b['classes'])
print('> model:', {k: _b.get(k) for k in ('task', 'classes', 'features', 'n_train', 'sklearn_ver')})

# every feature the model expects must be present in the enum features
_missing = [c for c in _fc if c not in MF_features_.columns]
assert not _missing, f'enum MF_features missing {len(_missing)} model feature cols, e.g. {_missing[:5]}'

# deploy: predict_proba -> per-class <cls>_pred columns + argmax label (rows aligned to MF_features)
_proba = _rf.predict_proba(MF_features_[_fc])
enum_pred = MF_features_[['compound', 'smiles']].copy()
for j, c in enumerate(_cls):
    enum_pred[f'{c}_pred'] = _proba[:, j]
enum_pred['pred_label'] = np.array(_cls)[_proba.argmax(1)]
# attach the reactant id (dedup to keep the 1:1 join from expanding rows)
enum_pred = enum_pred.merge(enum[['compound']].drop_duplicates('compound'),
                            on='compound', how='left')

print(f'\n> scored {len(enum_pred):,} enum compounds | columns: {list(enum_pred.columns)}')
print('> predicted-label distribution:'); print(enum_pred['pred_label'].value_counts().to_string())
# enum_pred.head()

> model: {'task': 'activity_multiclass_grouped', 'classes': ['High (>25)', 'Medium (11-25)', 'Silent', 'Single/Low (1-10)'], 'features': 'H236', 'n_train': 3342, 'sklearn_ver': '1.8.0'}

> scored 579 enum compounds | columns: ['compound', 'smiles', 'High (>25)_pred', 'Medium (11-25)_pred', 'Silent_pred', 'Single/Low (1-10)_pred', 'pred_label']
> predicted-label distribution:
pred_label
Single/Low (1-10)    483
Silent                88
High (>25)             6
Medium (11-25)         2


In [41]:
print(test_2['label'].value_counts().to_string())

label
Single/Low (1-10)    337
Silent                96
High (>25)            80
Medium (11-25)        66


In [45]:
## chemistry + label frame (the helper reads the 'label' column)
ML_data_ = MF_features.merge(test_2, on='compound').rename(columns={'activity': 'label'})
print('activity class sizes:'); print(ML_data_['label'].value_counts().to_string(), '\n')

model = RandomForestClassifier(n_estimators=400, class_weight='balanced', n_jobs=16, random_state=0)
trained_model, df_pred = ML_Class.run_K_Fold_Xval_multiclass_Classification(
    ML_data_, ID='compound', model=model, folds=5, col_to_rm=['compound', 'label','smiles'],
    v=False, ctf=0.5, impute_by_mean=False)

# report in ordinal order (Silent -> High) from the per-class *_pred probability columns
_order = [c for c in ['Silent', 'Single/Low (1-10)', 'Medium (11-25)', 'High (>25)']
          if c in set(df_pred['real_y'])]
print('\nRANDOM 5-fold CV  (RF, chemistry -> activity class)   (optimistic; analog leakage)\n')
fn.per_class_report(df_pred['real_y'].to_numpy(), df_pred['pred_y'].to_numpy(),
                    df_pred[[f'{c}_pred' for c in _order]].to_numpy(), _order)

# stratified-dummy macro-F1 floor
feat = [c for c in ML_data_.columns if c not in ('compound', 'label')]
X, y = ML_data_[feat].to_numpy(np.float32), ML_data_['label'].to_numpy()
_dummy = cross_val_predict(DummyClassifier(strategy='stratified', random_state=0), X, y,
                           cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0), n_jobs=16)
print(f'\n(stratified-dummy macro-F1 floor: {f1_score(y, _dummy, average="macro"):.2f})')

# expose results for the ROC-by-class plot cell below (act_proba columns align to act_classes)
act_classes = np.unique(df_pred['real_y'])
act_proba   = df_pred[[f'{c}_pred' for c in act_classes]].to_numpy()
act_y, act_pred, act_order = df_pred['real_y'].to_numpy(), df_pred['pred_y'].to_numpy(), _order


activity class sizes:
label
Single/Low (1-10)    907
Silent               273
High (>25)            81
Medium (11-25)        61 



  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:05<00:00,  1.12s/it]



RANDOM 5-fold CV  (RF, chemistry -> activity class)   (optimistic; analog leakage)

> Silent:	 Accuracy: 0.77, F1: 0.13, ROC_auc: 0.63, PR_auc: 0.28, MCC: 0.06
----------------------------------------------------------------------------------
> Single/Low (1-10):	 Accuracy: 0.67, F1: 0.80, ROC_auc: 0.60, PR_auc: 0.75, MCC: 0.05
----------------------------------------------------------------------------------
> Medium (11-25):	 Accuracy: 0.95, F1: 0.00, ROC_auc: 0.61, PR_auc: 0.07, MCC: -0.02
----------------------------------------------------------------------------------
> High (>25):	 Accuracy: 0.94, F1: 0.19, ROC_auc: 0.76, PR_auc: 0.27, MCC: 0.23
----------------------------------------------------------------------------------
----------------------------------------------------------------------------------
>> MACRO:	 Accuracy: 0.83, F1: 0.28, ROC_auc: 0.65, PR_auc: 0.34, MCC: 0.08


ValueError: could not convert string to float: 'O=C1NC[C@H]2C3=C(C=CC=C3)C[C@@H]1N2C(OCC1=CC=C(C#N)C=C1)=O'

In [47]:
df_pred
print(df_pred['pred_y'].value_counts().to_string())

pred_y
Single/Low (1-10)    1224
Silent                 71
High (>25)             16
Medium (11-25)         11
